# 3D toric code — telling a **first-order** from a **second-order** transition (NQS)

We contrast two cuts of the field phase diagram whose *order* QMC already knows:

| line | sweep | fixed | QMC order | mechanism |
|---|---|---|---|---|
| **1st** | $h_x$ | $h_z=0$ | first | flux/membrane condensation |
| **2nd** | $h_z$ | $h_x$ fixed | second (3D-Ising) | charge condensation |

**Primary diagnostic: the energy kink.** The field couples linearly, $H(h)=H_0-h\,M$ with $M=\sum_i\sigma^{(\mathrm{sweep})}_i$, so by Hellmann–Feynman $dE/dh=-\langle M\rangle=-N\,m$ and $d^2E/dh^2=-N\,dm/dh=-\chi$.

* **1st order:** $m$ jumps → $dE/dh$ discontinuous → $E(h)$ has a **kink**; $d^2E/dh^2$ peaks with height $\sim L^3$ (volume), width $\sim L^{-3}$.
* **2nd order:** $m$ continuous → $E$ smooth; the peak grows only weakly.

Every finite-$L$ ground state is analytic in $h$ — the kink is *emergent*, so we read the **trend with $L$**, not any single curve. Full physics + references: `notes/distinguishing_transition_order.md`.

In [ ]:
# ====================== 1 · CONFIG — the one cell to edit ======================
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt

ROOT = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"

# Each line: energy curves (pull with nersc/extract_energy.sh), FM + S2 order params.
# The 2nd-order comparison line can be any fixed-hx cut; 0.6 is mid-phase.
LINES = {
    "1st  (hx @ hz=0)": dict(
        order="first",  field="hx",
        energy=f"{ROOT}/energy_hz0.0",
        FM=f"{ROOT}/phase_hz0.0_memA0.5",   # membrane O_FM (magnetic sector)
        S2=f"{ROOT}/phase_hz0.0_s2plaq"),
    "2nd  (hz @ hx=0.6)": dict(
        order="second", field="hz",
        energy=f"{ROOT}/energy_hx0.6",
        FM=f"{ROOT}/phase_hx0.6_bulkR1",    # string O_FM (electric sector)
        S2=f"{ROOT}/phase_hx0.6_s2plaq"),
}

N_EDGES = lambda L: 3*L**2*(L-1)   # OBC cubic 3D-TC qubit count (3L^3 - 3L^2)
CMAP    = plt.cm.viridis
print("lines:", list(LINES))

## 2 · Data library

In [ ]:
# ====================== 2 · DATA LIBRARY ======================
def _load_dir(directory, keys):
    """Load every *.json in a dir keyed by L. `keys` maps output-name -> json-key
    (missing keys -> None). Returns {} if the dir is absent/empty."""
    recs = {}
    for jp in sorted(glob.glob(os.path.join(directory, "*.json"))):
        d = json.load(open(jp))
        rec = {"raw": d, "h": np.array(d["field"], float)}
        for name, k in keys.items():
            rec[name] = np.array(d[k], float) if d.get(k) is not None else None
        recs[int(d["L"])] = rec
    return recs

def load_line(spec):
    field = spec["field"]
    energy = _load_dir(spec["energy"], {"E": "E", "Espread": "E_spread",
                                          "m": ("mx" if field=="hx" else "mz"),
                                          "Vscore": "Vscore"})
    fm = _load_dir(spec["FM"], {"O": "O", "Oe": "Oe", "mz": "mz"})
    s2 = _load_dir(spec["S2"], {"S2": "S2", "S2e": "S2e"})
    return dict(field=field, order=spec["order"], energy=energy, FM=fm, S2=s2)

DATA = {name: load_line(spec) for name, spec in LINES.items()}

for name, D in DATA.items():
    eL = sorted(D["energy"]); fL = sorted(D["FM"]); sL = sorted(D["S2"])
    print(f"[{name}]  sweep={D['field']}  order={D['order']}")
    print(f"    energy L={eL or 'MISSING -> run nersc/extract_energy.sh + pull'}")
    print(f"    FM     L={fL}")
    print(f"    S2     L={sL}")

## 3 · Energy kink — the primary diagnostic

Three panels per line: **$E(h)$** (per site, so sizes overlay), its slope **$dE/dh$** with the Hellmann–Feynman cross-check $-N m$ overlaid, and the curvature **$d^2E/dh^2=-\chi$**. A first-order line shows a corner in $E/N$, a step in $dE/dh$, and a curvature spike that sharpens with $L$; a second-order line stays smooth. If energies aren't pulled yet, the slope panel still works on the second-order line via $dE/dh=-N\langle M_z\rangle$ from the FM json (on the first-order line $\langle M_z\rangle\approx0$ — wrong conjugate — so it needs the pull).

In [ ]:
# ====================== 3 · ENERGY CURVES + DERIVATIVES ======================
def energy_slope_from_mag(D):
    """Fallback dE/dh = -N<M_z> from the FM json when the energy pull is absent.
    Only valid on an hz-sweep: there M_z is the conjugate magnetization. On the
    hx-sweep (1st-order line) the conjugate is M_x (not in the FM extract; M_z~0),
    so we return {} -> that line HONESTLY needs the energy pull, no noise-fit."""
    if D["field"] != "hz":
        return {}
    out = {}
    for L, r in D["FM"].items():
        if r.get("mz") is not None:
            out[L] = (r["h"], -N_EDGES(L)*r["mz"])
    return out

for name, D in DATA.items():
    have_E = bool(D["energy"])
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    fig.suptitle(f"{name}   [{'energy pulled' if have_E else 'ENERGY NOT PULLED — slope from -N<Mz>'}]", fontsize=12)
    Ls = sorted(D["energy"]) if have_E else sorted(D["FM"])
    colors = CMAP(np.linspace(0, 0.85, max(1, len(Ls))))
    slope_fallback = energy_slope_from_mag(D)
    for L, c in zip(Ls, colors):
        if have_E:
            r = D["energy"][L]; h, E = r["h"], r["E"]; N = N_EDGES(L)
            o = np.argsort(h); h, E = h[o], E[o]
            ax[0].plot(h, E/N, "o-", ms=4, color=c, label=f"L={L}")
            dE = np.gradient(E, h)
            ax[1].plot(h, dE, "o-", ms=4, color=c, label=f"L={L}")
            if r.get("m") is not None:   # Hellmann-Feynman cross-check
                ax[1].plot(h, -N*r["m"][o], "x--", ms=5, color=c, alpha=0.6)
            ax[2].plot(h, np.gradient(dE, h), "o-", ms=4, color=c, label=f"L={L}")
        elif L in slope_fallback:
            h, dE = slope_fallback[L]; o = np.argsort(h)
            ax[1].plot(h[o], dE[o], "o-", ms=4, color=c, label=f"L={L}")
    ax[0].set(xlabel=f"${D['field']}$", ylabel="$E/N$", title="energy per site")
    ax[1].set(xlabel=f"${D['field']}$", ylabel="$dE/dh$",
              title="slope  (x-- = $-N\\langle M\\rangle$ check)")
    ax[2].set(xlabel=f"${D['field']}$", ylabel="$d^2E/dh^2=-\\chi$", title="curvature")
    for a in ax:
        if a.has_data(): a.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 4 · Quantifying the kink — **your call** ▶

The FSS below needs one scalar per $(L)$ that captures *how kinked* the curve is. There are a few defensible choices and the decision shapes the whole conclusion — so this is the cell to make yours:

* **`curvature_peak`** — $\max_h|d^2E/dh^2|$ (equivalently $N\max|dm/dh|$). Grows $\sim L^3$ (1st) vs weakly (2nd). Simplest; noise-sensitive (2nd derivative).
* **`slope_jump`** — fit a line to $dE/dh$ on each side of the peak and take the gap $\Delta(dE/dh)$. This is the *latent-heat analogue*; stays finite as $L\to\infty$ for 1st order, vanishes for 2nd. Most physical, needs a clean split point.
* **`peak_width`** — FWHM of the $|d^2E/dh^2|$ peak. Shrinks $\sim L^{-3}$ (1st) vs $\sim L^{-1/\nu}$ (2nd).

A reference `curvature_peak` is implemented so the notebook runs. **Edit / add your metric** in `kink_metric` — it returns `(scalar, h_peak)` and feeds §5.

In [ ]:
# ====================== 4 · KINK METRIC (EDIT ME) ======================
def _signal(h, y):
    """Return (h, dy/dh, d2y/dh2) on the given grid (sorted)."""
    o = np.argsort(h); h = h[o]; y = y[o]
    d1 = np.gradient(y, h); d2 = np.gradient(d1, h)
    return h, d1, d2

def kink_metric(h, E, kind="curvature_peak"):
    """One scalar measuring kink strength for a single (h, E) curve.
    Returns (value, h_peak). Feeds the finite-size scaling in section 5.

    TODO(you): the physics choice. `curvature_peak` is the reference. Consider
    implementing `slope_jump` (latent-heat analogue) or `peak_width`, and decide
    which best separates 1st from 2nd order over L=4..7 — see the markdown above."""
    hh, d1, d2 = _signal(h, E)
    if kind == "curvature_peak":
        i = int(np.argmax(np.abs(d2)))
        return float(np.abs(d2[i])), float(hh[i])
    # elif kind == "slope_jump":
    #     i = int(np.argmax(np.abs(d2)))          # split at the curvature peak
    #     left  = np.polyfit(hh[:i][-3:], d1[:i][-3:], 1)[0]  # ... your fit ...
    #     ...
    #     return abs(right_intercept - left_intercept), float(hh[i])
    # elif kind == "peak_width":
    #     ...  # FWHM of |d2|
    raise NotImplementedError(kind)

KIND = "curvature_peak"    # <- switch here once you add another metric

# quick smoke test on synthetic step vs smooth curves (does the metric separate them?)
_h = np.linspace(-1, 1, 41)
_step   = np.tanh(_h/0.03)      # near-discontinuous (mimics 1st order)
_smooth = np.tanh(_h/0.5)       # broad (mimics 2nd order)
print("metric on near-step :", round(kink_metric(_h, _step,   KIND)[0], 2))
print("metric on smooth    :", round(kink_metric(_h, _smooth, KIND)[0], 2),
      " (step should be >> smooth)")

## 5 · Finite-size scaling of the kink

Fit $\text{metric}(L)=a\,L^{p}$ on each line. A **first-order** kink metric that grows toward the volume exponent $p\!\to\!3$ (or a width that shrinks with $p\!\to\!-3$) vs a **second-order** exponent near $1/\nu\approx1.6$ (or $\gamma/\nu$) is the discriminant. With only $L=4\text{–}7$ the exponent is *suggestive*, not decisive — the more robust read is whether the metric **keeps climbing** (1st) or **saturates** (2nd).

In [ ]:
# ====================== 5 · FSS OF THE KINK METRIC ======================
def metric_vs_L(D, kind=KIND):
    """Prefer the pulled energy; else fall back to -N<Mz> (2nd-order line only)."""
    out = {}
    for L, r in D["energy"].items():
        out[L] = kink_metric(r["h"], r["E"], kind)
    if not out:
        for L, (h, dE) in energy_slope_from_mag(D).items():   # dE already = dE/dh
            hh, d1, _ = _signal(h, dE)                        # so one more deriv = d2E
            i = int(np.argmax(np.abs(d1)))
            out[L] = (float(np.abs(d1[i])), float(hh[i]))
    return out

fig, ax = plt.subplots(1, 1, figsize=(6.5, 5))
summary = {}
for (name, D), col in zip(DATA.items(), ("C3", "C0", "C2", "C1")):
    mv = metric_vs_L(D)
    if len(mv) < 2:
        print(f"[{name}] <2 sizes with a metric — pull energies to enable FSS."); continue
    Ls = np.array(sorted(mv)); vals = np.array([mv[L][0] for L in Ls])
    p, a = np.polyfit(np.log(Ls), np.log(vals), 1)   # slope p = power
    summary[name] = (p, Ls, vals)
    ax.plot(Ls, vals, "o", ms=9, color=col,
            label=f"{name}:  p={p:.2f}  ({D['order']})")
    xx = np.linspace(Ls.min(), Ls.max(), 50)
    ax.plot(xx, np.exp(a)*xx**p, "-", color=col, alpha=0.6)
ax.set(xscale="log", yscale="log", xlabel="$L$", ylabel=f"kink metric ({KIND})",
       title="metric $\\sim L^{p}$   (1st: p→3 / 2nd: p≈1/ν≈1.6)")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()
for name, (p, Ls, vals) in summary.items():
    print(f"  {name}: p = {p:.2f}   metric(L={Ls.tolist()}) = {np.round(vals,3).tolist()}")

## 6 · Cross-check with the order parameters (all three probes)

The energy kink is one probe; the FM ratio and Rényi-2 are independent ones. A genuine transition-order signal should agree across all three — disagreement flags an NQS artifact (a variational state stuck on one side of a coexistence barrier can *miss* a first-order jump, or optimization discontinuities can *fake* sharpness). Here we compare the finite-difference derivative peaks of $O_{FM}$ and $S_2$ (already in the jsons) alongside the energy metric.

In [ ]:
# ====================== 6 · ORDER-PARAMETER DERIVATIVE PEAKS ======================
def obs_peak_vs_L(D, which):
    key = "O" if which == "FM" else "S2"
    out = {}
    for L, r in D[which].items():
        if r.get(key) is None: continue
        hh, d1, _ = _signal(r["h"], r[key])
        i = int(np.argmax(np.abs(d1)))
        out[L] = (float(np.abs(d1[i])), float(hh[i]))
    return out

probes = [("energy", metric_vs_L), ("FM", lambda D: obs_peak_vs_L(D, "FM")),
          ("S2", lambda D: obs_peak_vs_L(D, "S2"))]
fig, ax = plt.subplots(1, len(DATA), figsize=(7*len(DATA), 5), squeeze=False)
for j, (name, D) in enumerate(DATA.items()):
    a = ax[0, j]
    for pname, fn in probes:
        mv = fn(D)
        if len(mv) < 2: continue
        Ls = np.array(sorted(mv)); vals = np.array([mv[L][0] for L in Ls])
        p = np.polyfit(np.log(Ls), np.log(vals/vals[0]), 1)[0]  # normalized, shared axis
        a.plot(Ls, vals/vals[0], "o-", ms=8, label=f"{pname}:  p={p:.2f}")
    a.set(xscale="log", yscale="log", xlabel="$L$", ylabel="peak / peak(L$_{min}$)",
          title=f"{name}  [{D['order']}]"); a.legend(fontsize=9)
plt.tight_layout(); plt.show()
print("Read: do the three probes AGREE on p (steep→1st, shallow→2nd)? "
      "Agreement = trust; a lone steep probe = suspect NQS artifact.")

## 7 · Verdict & honest caveats

**What would count as a first-order signal here (line 1):** a finite $m$/slope jump that does *not* shrink with $L$, a curvature-peak exponent $p$ climbing toward 3, and agreement across energy, $O_{FM}$, and $S_2$.

**What would count as second order (line 2):** smooth $E$, no slope step, shallow peak growth ($p\approx1/\nu$), consistent across probes.

**Caveats to keep in the paper:**
1. $L=4\text{–}7$ is <2× in $L$ — exponents are suggestive, not proof. Lead with the *qualitative* jump-vs-smooth contrast and the *sign of the trend*.
2. The first-order line is a single cut ($h_z=0$). A finite-$h_z$ first-order cut would show the order persists.
3. NQS variational bias: no tunneling across a coexistence barrier can hide a jump; cross-probe agreement is the guard. Gold-standard follow-ups not in these runs — bimodal energy histograms, Binder-cumulant negative dip (needs $\langle m^2\rangle,\langle m^4\rangle$), hysteresis from ordered/disordered inits.